# Transport

> Shared async HTTP transport.

In [ ]:
#| default_exp transport

## Imports

In [ ]:
#| export
import httpx, json

from fastspec.errors import ProtocolError
from fastspec.sse import aiter_sse

In [ ]:
#| hide
import os
from fastcore.test import *

### `AsyncTransport`

Thin async HTTP + SSE transport wrapper.

**Role In This Module**
- Public symbol in this module API surface.
- Centralizes request execution, content decoding, multipart behavior, and SSE streaming mechanics.

**Design Notes**
- Intentionally provider-agnostic — no provider logic here.
- Two public methods: `request` (single response) and `stream` (SSE async generator).
- `request` auto-decodes by content-type (JSON/text/binary); pass `raw=True` for the raw `httpx.Response`.
- `stream` parses SSE events, terminates on `[DONE]` (OpenAI) or `message_stop` (Anthropic), and yields parsed JSON dicts. Gemini ends stream automatically.
- Accepts an external `httpx.AsyncClient` for testability; tracks ownership via `_own_client` to avoid closing injected clients.

**Connections**
- Used by `OpenAPIClient` and therefore all provider clients.
- Depends on `aiter_sse` from `sse.py` for SSE framing.
- Raises `ProtocolError` from `errors.py` on malformed SSE data.

In [ ]:
#| export
class AsyncTransport:
    "Thin async transport wrapper over httpx."
    def __init__(self, *, timeout=60.0, client=None, base_headers=None):
        self._own_client = client is None
        self.client = client or httpx.AsyncClient(timeout=timeout)
        self.base_headers = base_headers or {}

    async def aclose(self):
        if self._own_client: await self.client.aclose()

    def _headers(self, headers=None):
        return {**self.base_headers, **(headers or {})}

    def _request_headers(self, headers=None, *, files=None):
        "Merge headers and drop explicit content-type for multipart uploads."
        h = self._headers(headers)
        if files is not None:
            # Let httpx compute multipart/form-data with boundary.
            for k in list(h):
                if k.lower() == "content-type": h.pop(k, None)
        return h

    @staticmethod
    def _decode(resp):
        "Decode response body using content type."
        if not resp.content: return None
        ctype = (resp.headers.get("content-type") or "").lower()
        if "application/json" in ctype or ctype.endswith("+json"): return resp.json()
        if ctype.startswith("text/") or "application/x-ndjson" in ctype: return resp.text
        return resp.content

    async def request(self, method, url, *, headers=None, params=None, json_data=None, data=None,
                      files=None, raw=False):
        "Execute a request and decode JSON/text/binary response."
        resp = await self.client.request(method, url, headers=self._request_headers(headers, files=files),
            params=params, json=json_data, data=data, files=files)
        try: resp.raise_for_status()
        except httpx.HTTPStatusError as e:
            e.args = (f"{e}\n{resp.text}",)
            raise
        return resp if raw else self._decode(resp)

    async def stream(self, method, url, *, headers=None, params=None,
                            json_data=None, data=None, files=None):
        async with self.client.stream(method, url, headers=self._request_headers(headers, files=files),
            params=params, json=json_data, data=data, files=files) as resp:
            try: resp.raise_for_status()
            except httpx.HTTPStatusError as e:
                try: await resp.aread()
                except Exception: pass
                e.args = (f"{e}\n{resp.text}",)
                raise
            async for event in aiter_sse(resp):
                if not event.data: continue
                if event.data == "[DONE]" or event.event == "message_stop": return
                try: raw = json.loads(event.data)
                except json.JSONDecodeError as e: raise ProtocolError(f"Invalid SSE JSON: {e}") from e
                if isinstance(raw, dict): yield raw
                else: raise ProtocolError(f"Expected SSE JSON object, got {type(raw).__name__}")

In [ ]:
t = AsyncTransport()
resp = await t.request("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json_data={"model": "gpt-4o-mini", "max_tokens": 5, "messages": [{"role": "user", "content": "Say hi"}]})
resp

{'id': 'chatcmpl-DSI6BC08af1l5QcvQdVIAfci4sQMN',
 'object': 'chat.completion',
 'created': 1775635127,
 'model': 'gpt-4o-mini-2024-07-18',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Hi! How can I',
    'refusal': None,
    'annotations': []},
   'logprobs': None,
   'finish_reason': 'length'}],
 'usage': {'prompt_tokens': 9,
  'completion_tokens': 5,
  'total_tokens': 14,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 0,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': 'fp_9a00d2658e'}

In [ ]:
chunks = []
async for chunk in t.stream("POST", "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json_data={"model": "gpt-4o-mini", "stream": True, "max_tokens": 5, "messages": [{"role": "user", "content": "Say hi"}]}):
    chunks.append(chunk)
chunks[:3]

[{'id': 'chatcmpl-DSI6CyUY8WRGIzhPsQ9sjlpYZVEYG',
  'object': 'chat.completion.chunk',
  'created': 1775635128,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_9a00d2658e',
  'choices': [{'index': 0,
    'delta': {'role': 'assistant', 'content': '', 'refusal': None},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': 'q4pc77'},
 {'id': 'chatcmpl-DSI6CyUY8WRGIzhPsQ9sjlpYZVEYG',
  'object': 'chat.completion.chunk',
  'created': 1775635128,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_9a00d2658e',
  'choices': [{'index': 0,
    'delta': {'content': 'Hi'},
    'logprobs': None,
    'finish_reason': None}],
  'obfuscation': 'FN7PfG'},
 {'id': 'chatcmpl-DSI6CyUY8WRGIzhPsQ9sjlpYZVEYG',
  'object': 'chat.completion.chunk',
  'created': 1775635128,
  'model': 'gpt-4o-mini-2024-07-18',
  'service_tier': 'default',
  'system_fingerprint': 'fp_9a00d2658e',
  'choices': [{'index': 0,
    

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()